================================================================================
DISASTER HUMAN DETECTION ABLATION STUDY
Comprehensive Evaluation: YOLOv11m Baseline vs ECA-Modified Architecture
================================================================================

Dataset: C2A (Disaster Scenarios)
Task: Single-class person detection
Models: YOLOv11m-C2PSA (Baseline) vs YOLOv11m-ECA (Modified)

Metrics Suite:
✓ IoU-based TP/FP/FN calculation
✓ Precision, Recall, F1, F2 (recall-weighted)
✓ Size-based performance (tiny/small/medium/large objects)
✓ Confidence calibration analysis
✓ Optimal threshold recommendation
✓ Failure mode analysis
✓ Speed vs resolution benchmarking
✓ Comprehensive comparative reports

INSTRUCTIONS:
1. Copy each cell below into separate Kaggle notebook cells
2. Run cells sequentially (DO NOT skip cells)
3. Training takes ~45min on T4 GPU
4. Evaluation takes ~5-10min
5. All results auto-exported to Excel/PNG/TXT

================================================================================
"""


In [1]:
!pip install -q -U ultralytics timm thop pandas matplotlib openpyxl scikit-learn
print("✓ All dependencies installed successfully")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 90.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires matplotlib<=3.10,>=3.5, but you have matplotlib 3.10.8 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have 

In [1]:
from ultralytics import YOLO
import yaml

model = YOLO("yolo11m.pt")
model.info()

cfg = model.model.yaml
with open("yolov11m_original.yaml", "w") as f:
    yaml.dump(cfg, f)
with open("yolov11m_eca.yaml", "w") as f:
    yaml.dump(cfg, f)

print("✓ Extracted YOLOv11m architecture to yolov11m_eca.yaml")
print(f"✓ Model has {len(cfg['backbone'])} backbone layers")

d:\Anaconda3\Lib\site-packages\ultralytics\nn\tasks.py:567: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(file, map_location='cpu'), file  # load


FileNotFoundError: [Errno 2] No such file or directory: 'yolo11m.pt'

In [3]:
import yaml
import shutil

shutil.copy("yolov11m_eca.yaml", "yolov11m_eca_backup.yaml")

with open("yolov11m_eca.yaml", "r") as f:
    cfg = yaml.safe_load(f)

replaced_count = 0
for i, layer in enumerate(cfg["backbone"]):
    if len(layer) >= 3 and layer[2] == "C2PSA":
        channels = layer[3][0] if len(layer) > 3 and isinstance(layer[3], list) else 256
        cfg["backbone"][i] = [-1, 1, "ECA", [channels]]
        replaced_count += 1
        print(f"Layer {i}: C2PSA({channels} channels) → ECA({channels} channels)")

assert replaced_count > 0, "❌ FATAL: No C2PSA layers found!"

with open("yolov11m_eca.yaml", "w") as f:
    yaml.dump(cfg, f)

print(f"\n✓ Successfully replaced {replaced_count} C2PSA layer(s) with ECA")


Layer 10: C2PSA(1024 channels) → ECA(1024 channels)

✓ Successfully replaced 1 C2PSA layer(s) with ECA


In [4]:
import torch
import torch.nn as nn
import math

class ECA(nn.Module):
    """
    Efficient Channel Attention - Paper Implementation
    Adaptive kernel size based on channel count
    """
    def __init__(self, channels, gamma=2, b=1):
        super().__init__()
        t = int(abs(math.log2(channels) / gamma + b / gamma))
        k = t if t % 2 else t + 1
        
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=(k - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
        self.channels = channels
        self.kernel_size = k
    
    def forward(self, x):
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(-1, -2)
        y = self.conv(y)
        y = self.sigmoid(y).transpose(-1, -2).unsqueeze(-1)
        return x * y.expand_as(x)
    
    def __repr__(self):
        return f"ECA(channels={self.channels}, kernel_size={self.kernel_size})"

# Test ECA
for channels in [64, 128, 256, 512]:
    eca = ECA(channels)
    x = torch.randn(2, channels, 32, 32)
    out = eca(x)
    assert out.shape == x.shape
    print(f"✓ ECA({channels}): kernel_size={eca.kernel_size}")

print("\n✓ ECA module tested successfully")

✓ ECA(64): kernel_size=3
✓ ECA(128): kernel_size=5
✓ ECA(256): kernel_size=5
✓ ECA(512): kernel_size=5

✓ ECA module tested successfully


In [5]:
import ultralytics.nn.modules as modules
import ultralytics.nn.tasks as tasks

modules.ECA = ECA
tasks.ECA = ECA

assert "ECA" in dir(modules)
assert "ECA" in dir(tasks)

print("✓ ECA registered in ultralytics.nn.modules")
print("✓ ECA registered in ultralytics.nn.tasks")

✓ ECA registered in ultralytics.nn.modules
✓ ECA registered in ultralytics.nn.tasks


In [6]:
dataset_yaml_content = """train: /kaggle/input/c2a-dataset/C2A_Dataset/new_dataset3/train/images
val: /kaggle/input/c2a-dataset/C2A_Dataset/new_dataset3/val/images
test: /kaggle/input/c2a-dataset/C2A_Dataset/new_dataset3/test/images

nc: 1
names: ['person']
"""

with open("c2a.yaml", "w") as f:
    f.write(dataset_yaml_content)

print("✓ Dataset configuration saved to c2a.yaml")

✓ Dataset configuration saved to c2a.yaml


In [2]:
from ultralytics import YOLO

print("="*80)
print("TRAINING BASELINE YOLOv11m")
print("="*80)

baseline = YOLO("yolo11m.pt")
baseline.train(
    data="c2a.yaml",
    epochs=70,
    imgsz=640,
    batch=16,
    device=0,
    name="yolo11m_baseline",
    patience=5,
    save=True,
    save_period=-1,
    verbose=True,
    plots=True
)

print("\n✓ Baseline training complete")

TRAINING BASELINE YOLOv11m


FileNotFoundError: [Errno 2] No such file or directory: 'yolo11m.pt'

In [ ]:
from ultralytics import YOLO

print("="*80)
print("TRAINING ECA-MODIFIED YOLOv11m")
print("="*80)

eca_model = YOLO("yolov11m_eca.yaml")
eca_model.load("yolo11m.pt")

print("✓ ECA layers initialized")

eca_model.train(
    data="c2a.yaml",
    epochs=70,
    imgsz=640,
    batch=16,
    device=0,
    name="yolo11m_eca",
    patience=5,
    save=True,
    save_period=-1,
    verbose=True,
    plots=True
)

print("\n✓ ECA model training complete")

TRAINING ECA-MODIFIED YOLOv11m
Transferred 607/608 items from pretrained weights
✓ ECA layers initialized
Ultralytics 8.4.11 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c2a.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov11m_eca.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11m_eca, nbs=64, nms=F

In [ ]:
from ultralytics import YOLO

baseline = YOLO("runs/detect/yolo11m_baseline/weights/best.pt")
eca_model = YOLO("runs/detect/yolo11m_eca/weights/best.pt")

print("Baseline Validation:")
base_metrics = baseline.val(data="c2a.yaml", split="val")

print("\nECA Validation:")
eca_metrics = eca_model.val(data="c2a.yaml", split="val")

print("\n✓ Quick validation complete")

Baseline Validation:
Ultralytics 8.4.11 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 729.2±540.7 MB/s, size: 796.1 KB)
val: Scanning /kaggle/input/c2a-dataset/C2A_Dataset/new_dataset3/val/labels... 2043 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2043/2043 728.2it/s 2.8s
WARNING ⚠️ val: Cache directory /kaggle/input/c2a-dataset/C2A_Dataset/new_dataset3/val is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 128/128 1.6it/s 1:18
                   all       2043      72123      0.883      0.804      0.856      0.622
Speed: 0.9ms preprocess, 30.6ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /kaggle/working/runs/detect/val

ECA Validation:
Ultralytics 8.4.11 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913Mi

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = "/kaggle/working"
EXCEL_DIR = f"{BASE_DIR}/excel_reports"
PLOT_DIR = f"{BASE_DIR}/plots"
REPORT_DIR = f"{BASE_DIR}/benchmark_reports"

for d in [EXCEL_DIR, PLOT_DIR, REPORT_DIR]:
    os.makedirs(d, exist_ok=True)

BASELINE_RUN = "runs/detect/yolo11m_baseline"
ECA_RUN = "runs/detect/yolo11m_eca"

baseline_df = pd.read_csv(f"{BASELINE_RUN}/results.csv")
eca_df = pd.read_csv(f"{ECA_RUN}/results.csv")

baseline_df.to_excel(f"{EXCEL_DIR}/yolo11m_baseline_training.xlsx", index=False)
eca_df.to_excel(f"{EXCEL_DIR}/yolo11m_eca_training.xlsx", index=False)

def plot_losses(df, tag):
    plt.figure(figsize=(12, 6))
    for k in ["box", "cls", "dfl"]:
        if f"train/{k}_loss" in df.columns:
            plt.plot(df["epoch"], df[f"train/{k}_loss"], label=f"Train {k}", alpha=0.7)
            plt.plot(df["epoch"], df[f"val/{k}_loss"], label=f"Val {k}", linestyle='--')
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title(f"{tag} - Loss Curves")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/{tag}_losses.png", dpi=300)
    plt.close()

plot_losses(baseline_df, "yolo11m_baseline")
plot_losses(eca_df, "yolo11m_eca")

print("✓ Training curves exported")

✓ Training curves exported


In [3]:
def plot_metrics(df, tag):
    plt.figure(figsize=(14, 6))
    for col in ["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]:
        if col in df.columns:
            label = col.replace("metrics/", "").replace("(B)", "")
            plt.plot(df["epoch"], df[col], label=label, marker='o', markersize=3)
    plt.xlabel("Epoch"); plt.ylabel("Metric Value"); plt.title(f"{tag} - Validation Metrics")
    plt.legend(); plt.grid(True, alpha=0.3); plt.ylim([0, 1.05]); plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/{tag}_metrics.png", dpi=300)
    plt.close()

plot_metrics(baseline_df, "yolo11m_baseline")
plot_metrics(eca_df, "yolo11m_eca")

print("✓ Metric evolution plots saved")


NameError: name 'baseline_df' is not defined

In [12]:
from datetime import datetime

def write_training_report(tag, df):
    best_idx = df["metrics/mAP50(B)"].idxmax() if "metrics/mAP50(B)" in df.columns else len(df)-1
    best = df.loc[best_idx]
    p, r = best.get("metrics/precision(B)", 0), best.get("metrics/recall(B)", 0)
    f1 = 2*p*r/(p+r+1e-9)
    
    report = f"""
{'='*80}
TRAINING SUMMARY: {tag}
{'='*80}
Best Epoch: {int(best['epoch'])}
Precision: {p:.4f} | Recall: {r:.4f} | F1: {f1:.4f}
mAP@0.5: {best.get('metrics/mAP50(B)', 0):.4f}
mAP@0.5:0.95: {best.get('metrics/mAP50-95(B)', 0):.4f}
{'='*80}
"""
    with open(f"{REPORT_DIR}/{tag}_training_summary.txt", "w") as f:
        f.write(report)
    print(f"✓ Report: {tag}_training_summary.txt")

write_training_report("yolo11m_baseline", baseline_df)
write_training_report("yolo11m_eca", eca_df)

print("\n✓ Training phase complete")

✓ Report: yolo11m_baseline_training_summary.txt
✓ Report: yolo11m_eca_training_summary.txt

✓ Training phase complete


In [13]:
import os
import time
import cv2
import numpy as np
import torch
from ultralytics.utils.metrics import box_iou

def get_image_list(img_dir):
    return sorted([f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".png", ".jpeg"))])

def parse_yolo_label(label_path, img_w, img_h):
    if not os.path.exists(label_path):
        return torch.empty((0, 4))
    boxes = []
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            _, xc, yc, w, h = map(float, parts[:5])
            xc *= img_w; yc *= img_h; w *= img_w; h *= img_h
            boxes.append([xc-w/2, yc-h/2, xc+w/2, yc+h/2])
    return torch.tensor(boxes) if boxes else torch.empty((0, 4))

def match_predictions_to_gt(pred_boxes, gt_boxes, iou_thresh=0.5):
    if len(pred_boxes) == 0:
        return 0, 0, len(gt_boxes)
    if len(gt_boxes) == 0:
        return 0, len(pred_boxes), 0
    ious = box_iou(pred_boxes, gt_boxes)
    matched_gt = set()
    tp = 0
    for pred_idx in range(len(pred_boxes)):
        if ious.shape[1] == 0:
            break
        max_iou, max_idx = ious[pred_idx].max(dim=0)
        if max_iou >= iou_thresh and max_idx.item() not in matched_gt:
            tp += 1
            matched_gt.add(max_idx.item())
    return tp, len(pred_boxes) - tp, len(gt_boxes) - tp

def categorize_by_size(boxes):
    if len(boxes) == 0:
        return {'tiny': torch.zeros(0, dtype=torch.bool), 'small': torch.zeros(0, dtype=torch.bool),
                'medium': torch.zeros(0, dtype=torch.bool), 'large': torch.zeros(0, dtype=torch.bool)}
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return {'tiny': areas < 16**2, 'small': (areas >= 16**2) & (areas < 32**2),
            'medium': (areas >= 32**2) & (areas < 96**2), 'large': areas >= 96**2}

print("✓ Helper functions loaded")

✓ Helper functions loaded


In [14]:
import pandas as pd
from tqdm import tqdm

def evaluate_model_comprehensive(model, model_name, img_dir, lbl_dir, split_name="test", n_images=None, conf=0.25):
    print(f"\n{'='*80}\nEVALUATING: {model_name} on {split_name.upper()}\n{'='*80}")
    image_list = get_image_list(img_dir)
    if n_images:
        image_list = image_list[:min(n_images, len(image_list))]
    
    results = []
    inference_times = []
    total_tp, total_fp, total_fn = 0, 0, 0
    size_stats = {'tiny': {'tp': 0, 'fn': 0}, 'small': {'tp': 0, 'fn': 0},
                  'medium': {'tp': 0, 'fn': 0}, 'large': {'tp': 0, 'fn': 0}}
    
    for img_file in tqdm(image_list, desc=f"{model_name}-{split_name}"):
        img_path = f"{img_dir}/{img_file}"
        lbl_path = f"{lbl_dir}/{img_file.rsplit('.', 1)[0]}.txt"
        img = cv2.imread(img_path)
        if img is None:
            continue
        img_h, img_w = img.shape[:2]
        gt_boxes = parse_yolo_label(lbl_path, img_w, img_h)
        
        start = time.time()
        preds = model.predict(img_path, conf=conf, verbose=False)
        infer_ms = (time.time() - start) * 1000
        inference_times.append(infer_ms)
        
        if len(preds[0].boxes) > 0:
            pred_boxes = preds[0].boxes.xyxy.cpu()
            avg_conf = float(preds[0].boxes.conf.cpu().mean())
        else:
            pred_boxes = torch.empty((0, 4))
            avg_conf = 0.0
        
        tp, fp, fn = match_predictions_to_gt(pred_boxes, gt_boxes, 0.5)
        total_tp += tp; total_fp += fp; total_fn += fn
        
        if len(gt_boxes) > 0:
            for size_cat, mask in categorize_by_size(gt_boxes).items():
                if mask.sum() > 0:
                    size_tp, _, size_fn = match_predictions_to_gt(pred_boxes, gt_boxes[mask], 0.5)
                    size_stats[size_cat]['tp'] += size_tp
                    size_stats[size_cat]['fn'] += size_fn
        
        precision = tp / (tp + fp + 1e-9)
        recall = tp / (tp + fn + 1e-9)
        f1 = 2 * precision * recall / (precision + recall + 1e-9)
        f2 = 5 * precision * recall / (4 * precision + recall + 1e-9)
        
        results.append({'Image': img_file, 'GT_Boxes': len(gt_boxes), 'Pred_Boxes': len(pred_boxes),
                       'TP': tp, 'FP': fp, 'FN': fn, 'Precision': precision, 'Recall': recall,
                       'F1': f1, 'F2': f2, 'Avg_Confidence': avg_conf, 'Inference_ms': infer_ms})
    
    df = pd.DataFrame(results)
    df.to_excel(f"{EXCEL_DIR}/{model_name}_{split_name}_detailed.xlsx", index=False)
    
    overall_precision = total_tp / (total_tp + total_fp + 1e-9)
    overall_recall = total_tp / (total_tp + total_fn + 1e-9)
    overall_f1 = 2 * overall_precision * overall_recall / (overall_precision + overall_recall + 1e-9)
    overall_f2 = 5 * overall_precision * overall_recall / (4 * overall_precision + overall_recall + 1e-9)
    
    size_recalls = {f"{cat}_recall": stats['tp'] / (stats['tp'] + stats['fn']) 
                   if stats['tp'] + stats['fn'] > 0 else 0.0 for cat, stats in size_stats.items()}
    
    summary = {'Model': model_name, 'Split': split_name, 'Images': len(image_list),
               'Total_GT': total_tp + total_fn, 'Total_Pred': total_tp + total_fp,
               'TP': total_tp, 'FP': total_fp, 'FN': total_fn,
               'Precision': overall_precision, 'Recall': overall_recall,
               'F1': overall_f1, 'F2': overall_f2, **size_recalls,
               'Avg_Inference_ms': np.mean(inference_times), 'Std_Inference_ms': np.std(inference_times)}
    
    print(f"\nSUMMARY: P={overall_precision:.4f} R={overall_recall:.4f} F1={overall_f1:.4f}")
    return df, summary, inference_times

print("✓ Evaluation function loaded")

✓ Evaluation function loaded


In [15]:
def visualize_predictions(model, img_dir, lbl_dir, model_name, split="test", n_images=15, conf=0.25):
    images = get_image_list(img_dir)[:n_images]
    cols = 5
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(20, 4 * rows))
    axes = axes.flatten() if isinstance(axes, np.ndarray) else [axes]
    
    for i, img_file in enumerate(images):
        img_path = f"{img_dir}/{img_file}"
        lbl_path = f"{lbl_dir}/{img_file.rsplit('.', 1)[0]}.txt"
        gt_count = len(open(lbl_path).readlines()) if os.path.exists(lbl_path) else 0
        preds = model.predict(img_path, conf=conf, verbose=False)
        img_plot = preds[0].plot()
        axes[i].imshow(cv2.cvtColor(img_plot, cv2.COLOR_BGR2RGB))
        axes[i].axis("off")
        pred_count = len(preds[0].boxes)
        color = 'green' if pred_count == gt_count else 'orange' if abs(pred_count - gt_count) <= 1 else 'red'
        axes[i].set_title(f"GT: {gt_count} | Pred: {pred_count}", fontsize=10, color=color, fontweight='bold')
    
    for j in range(len(images), len(axes)):
        axes[j].axis("off")
    
    plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/{model_name}_{split}_predictions.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Visualization: {model_name}_{split}_predictions.png")

print("✓ Visualization function loaded")

✓ Visualization function loaded


In [16]:
def generate_ablation_comparison(baseline_summary, eca_summary, split_name):
    metrics = [("Precision", "Precision"), ("Recall", "Recall"), ("F1", "F1"), ("F2", "F2"),
               ("Small Object Recall", "small_recall"), ("Avg Inference (ms)", "Avg_Inference_ms")]
    comparison_data = []
    for display_name, key in metrics:
        baseline_val = baseline_summary.get(key, 0.0)
        eca_val = eca_summary.get(key, 0.0)
        comparison_data.append({'Metric': display_name, 'Baseline': round(baseline_val, 4),
                               'ECA': round(eca_val, 4), 'Δ': round(eca_val - baseline_val, 4),
                               'Δ (%)': round((eca_val - baseline_val) / (baseline_val + 1e-9) * 100, 2)})
    
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df.to_excel(f"{EXCEL_DIR}/ablation_comparison_{split_name}.xlsx", index=False)
    print(f"\n{'='*80}\nABLATION COMPARISON - {split_name.upper()}\n{'='*80}")
    print(comparison_df.to_string(index=False))
    print('='*80)
    return comparison_df

def generate_comprehensive_text_report(baseline_summary, eca_summary, split_name):
    report = f"""
{'='*80}
ABLATION STUDY: {split_name.upper()} SET
{'='*80}
BASELINE: P={baseline_summary['Precision']:.4f} R={baseline_summary['Recall']:.4f} F1={baseline_summary['F1']:.4f}
ECA:      P={eca_summary['Precision']:.4f} R={eca_summary['Recall']:.4f} F1={eca_summary['F1']:.4f}

DELTA: ΔP={eca_summary['Precision']-baseline_summary['Precision']:.4f} ΔR={eca_summary['Recall']-baseline_summary['Recall']:.4f} ΔF1={eca_summary['F1']-baseline_summary['F1']:.4f}

RECOMMENDATION: {'ECA' if eca_summary['Recall'] > baseline_summary['Recall'] else 'Baseline'} (Recall priority for disaster detection)
{'='*80}
"""
    with open(f"{REPORT_DIR}/ablation_study_{split_name}.txt", "w") as f:
        f.write(report)
    print(f"✓ Report: ablation_study_{split_name}.txt")
    return report

print("✓ Comparison functions loaded")

✓ Comparison functions loaded


In [17]:
from sklearn.metrics import precision_recall_curve, average_precision_score

def analyze_confidence_calibration(results_df, model_name, split_name, n_bins=10):
    df_with_preds = results_df[results_df['Pred_Boxes'] > 0].copy()
    if len(df_with_preds) == 0:
        return None, None
    
    bins = np.linspace(0, 1, n_bins + 1)
    calibration_data = []
    for i in range(n_bins):
        bin_mask = (df_with_preds['Avg_Confidence'] >= bins[i]) & (df_with_preds['Avg_Confidence'] < bins[i+1])
        if bin_mask.sum() > 0:
            calibration_data.append({
                'Bin': f"{bins[i]:.2f}-{bins[i+1]:.2f}",
                'Avg_Conf': df_with_preds.loc[bin_mask, 'Avg_Confidence'].mean(),
                'Avg_Prec': df_with_preds.loc[bin_mask, 'Precision'].mean(),
                'Count': bin_mask.sum(),
                'Gap': abs(df_with_preds.loc[bin_mask, 'Avg_Confidence'].mean() - df_with_preds.loc[bin_mask, 'Precision'].mean())
            })
    
    if not calibration_data:
        return None, None
    cal_df = pd.DataFrame(calibration_data)
    ece = np.average(cal_df['Gap'], weights=cal_df['Count'])
    cal_df.to_excel(f"{EXCEL_DIR}/{model_name}_{split_name}_calibration.xlsx", index=False)
    
    plt.figure(figsize=(10, 6))
    plt.plot([0, 1], [0, 1], 'k--', label='Perfect')
    plt.scatter(cal_df['Avg_Conf'], cal_df['Avg_Prec'], s=cal_df['Count']*10, alpha=0.6)
    plt.xlabel('Confidence'); plt.ylabel('Precision')
    plt.title(f'{model_name} Calibration (ECE={ece:.4f})')
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/{model_name}_{split_name}_calibration.png", dpi=150)
    plt.close()
    
    print(f"  ECE: {ece:.4f}")
    return cal_df, ece

def analyze_failure_modes(results_df, model_name, split_name):
    dangerous = results_df[(results_df['Avg_Confidence'] > 0.7) & (results_df['Recall'] < 0.5)]
    high_fn = results_df[results_df['FN'] > 3]
    high_fp = results_df[results_df['FP'] > 3]
    
    if len(dangerous) > 0:
        dangerous.to_excel(f"{EXCEL_DIR}/{model_name}_{split_name}_dangerous_errors.xlsx", index=False)
    
    print(f"  Dangerous errors: {len(dangerous)}, High FN: {len(high_fn)}, High FP: {len(high_fp)}")
    return {'Dangerous': len(dangerous), 'High_FN': len(high_fn), 'High_FP': len(high_fp)}

def benchmark_speed_vs_resolution(model, sample_img_path, model_name):
    results = []
    for imgsz in [320, 480, 640, 800]:
        times = [time.time() for _ in range(10)]
        for i in range(10):
            start = time.time()
            _ = model.predict(sample_img_path, imgsz=imgsz, verbose=False)
            times[i] = (time.time() - start) * 1000
        results.append({'Resolution': imgsz, 'Avg_ms': np.mean(times), 'FPS': 1000/np.mean(times)})
    
    speed_df = pd.DataFrame(results)
    speed_df.to_excel(f"{EXCEL_DIR}/{model_name}_speed_vs_resolution.xlsx", index=False)
    print(speed_df.to_string(index=False))
    return speed_df

print("✓ Advanced metrics loaded")

✓ Advanced metrics loaded


In [18]:
from ultralytics import YOLO

DATASET_ROOT = "/kaggle/input/c2a-dataset/C2A_Dataset/new_dataset3"
TEST_IMG_DIR = f"{DATASET_ROOT}/test/images"
TEST_LBL_DIR = f"{DATASET_ROOT}/test/labels"
VAL_IMG_DIR = f"{DATASET_ROOT}/val/images"
VAL_LBL_DIR = f"{DATASET_ROOT}/val/labels"

print("\n" + "="*80 + "\nPHASE 1: TEST SET (30 images)\n" + "="*80)

baseline_model = YOLO("runs/detect/yolo11m_baseline/weights/best.pt")
eca_model = YOLO("runs/detect/yolo11m_eca/weights/best.pt")

baseline_test_df, baseline_test_summary, _ = evaluate_model_comprehensive(
    baseline_model, "yolo11m_baseline", TEST_IMG_DIR, TEST_LBL_DIR, "test", n_images=30
)
eca_test_df, eca_test_summary, _ = evaluate_model_comprehensive(
    eca_model, "yolo11m_eca", TEST_IMG_DIR, TEST_LBL_DIR, "test", n_images=30
)

visualize_predictions(baseline_model, TEST_IMG_DIR, TEST_LBL_DIR, "yolo11m_baseline", "test", 15)
visualize_predictions(eca_model, TEST_IMG_DIR, TEST_LBL_DIR, "yolo11m_eca", "test", 15)

test_comparison = generate_ablation_comparison(baseline_test_summary, eca_test_summary, "test")
test_report = generate_comprehensive_text_report(baseline_test_summary, eca_test_summary, "test")


PHASE 1: TEST SET (30 images)

EVALUATING: yolo11m_baseline on TEST


yolo11m_baseline-test: 100%|██████████| 30/30 [00:02<00:00, 10.69it/s]



SUMMARY: P=0.8304 R=0.8529 F1=0.8415

EVALUATING: yolo11m_eca on TEST


yolo11m_eca-test: 100%|██████████| 30/30 [00:01<00:00, 17.64it/s]



SUMMARY: P=0.8342 R=0.8458 F1=0.8400
✓ Visualization: yolo11m_baseline_test_predictions.png
✓ Visualization: yolo11m_eca_test_predictions.png

ABLATION COMPARISON - TEST
             Metric  Baseline     ECA        Δ  Δ (%)
          Precision    0.8304  0.8342   0.0037   0.45
             Recall    0.8529  0.8458  -0.0070  -0.82
                 F1    0.8415  0.8400  -0.0015  -0.18
                 F2    0.8483  0.8435  -0.0048  -0.56
Small Object Recall    0.8835  0.8835   0.0000   0.00
 Avg Inference (ms)   56.8338 42.7928 -14.0411 -24.71
✓ Report: ablation_study_test.txt


In [19]:
print("\n" + "="*80 + "\nPHASE 2: VALIDATION SET (100 images)\n" + "="*80)

baseline_val_df, baseline_val_summary, _ = evaluate_model_comprehensive(
    baseline_model, "yolo11m_baseline", VAL_IMG_DIR, VAL_LBL_DIR, "val", n_images=100
)
eca_val_df, eca_val_summary, _ = evaluate_model_comprehensive(
    eca_model, "yolo11m_eca", VAL_IMG_DIR, VAL_LBL_DIR, "val", n_images=100
)

visualize_predictions(baseline_model, VAL_IMG_DIR, VAL_LBL_DIR, "yolo11m_baseline", "val", 15)
visualize_predictions(eca_model, VAL_IMG_DIR, VAL_LBL_DIR, "yolo11m_eca", "val", 15)

val_comparison = generate_ablation_comparison(baseline_val_summary, eca_val_summary, "val")
val_report = generate_comprehensive_text_report(baseline_val_summary, eca_val_summary, "val")


PHASE 2: VALIDATION SET (100 images)

EVALUATING: yolo11m_baseline on VAL


yolo11m_baseline-val: 100%|██████████| 100/100 [00:06<00:00, 14.49it/s]



SUMMARY: P=0.8075 R=0.7944 F1=0.8009

EVALUATING: yolo11m_eca on VAL


yolo11m_eca-val: 100%|██████████| 100/100 [00:06<00:00, 15.73it/s]



SUMMARY: P=0.8047 R=0.7967 F1=0.8007
✓ Visualization: yolo11m_baseline_val_predictions.png
✓ Visualization: yolo11m_eca_val_predictions.png

ABLATION COMPARISON - VAL
             Metric  Baseline     ECA       Δ  Δ (%)
          Precision    0.8075  0.8047 -0.0028  -0.35
             Recall    0.7944  0.7967  0.0023   0.29
                 F1    0.8009  0.8007 -0.0002  -0.03
                 F2    0.7970  0.7983  0.0013   0.16
Small Object Recall    0.8804  0.8820  0.0016   0.18
 Avg Inference (ms)   44.8955 40.2604 -4.6351 -10.32
✓ Report: ablation_study_val.txt


In [20]:
print("\n" + "="*80 + "\nPHASE 3: ADVANCED METRICS\n" + "="*80)

print("\nConfidence Calibration:")
baseline_cal, baseline_ece = analyze_confidence_calibration(baseline_test_df, "yolo11m_baseline", "test")
eca_cal, eca_ece = analyze_confidence_calibration(eca_test_df, "yolo11m_eca", "test")

print("\nFailure Modes:")
baseline_failures = analyze_failure_modes(baseline_test_df, "yolo11m_baseline", "test")
eca_failures = analyze_failure_modes(eca_test_df, "yolo11m_eca", "test")

print("\nSpeed Benchmarks:")
test_images = get_image_list(TEST_IMG_DIR)
sample_img = f"{TEST_IMG_DIR}/{test_images[0]}"
print("Baseline:")
baseline_speed = benchmark_speed_vs_resolution(baseline_model, sample_img, "yolo11m_baseline")
print("\nECA:")
eca_speed = benchmark_speed_vs_resolution(eca_model, sample_img, "yolo11m_eca")


PHASE 3: ADVANCED METRICS

Confidence Calibration:
  ECE: 0.0991
  ECE: 0.1002

Failure Modes:
  Dangerous errors: 0, High FN: 19, High FP: 19
  Dangerous errors: 0, High FN: 18, High FP: 20

Speed Benchmarks:
Baseline:
 Resolution    Avg_ms       FPS
        320 25.752544 38.831114
        480 27.783775 35.992229
        640 30.848813 32.416158
        800 47.454429 21.072849

ECA:
 Resolution    Avg_ms       FPS
        320 17.541862 57.006493
        480 20.112967 49.719168
        640 31.016183 32.241234
        800 46.394300 21.554372


In [21]:
print("\n" + "="*80 + "\nFINAL SUMMARY\n" + "="*80)

all_summaries = pd.DataFrame([baseline_test_summary, eca_test_summary, baseline_val_summary, eca_val_summary])
all_summaries.to_excel(f"{EXCEL_DIR}/complete_evaluation_summary.xlsx", index=False)

master_report = f"""
{'='*80}
DISASTER HUMAN DETECTION: ABLATION STUDY COMPLETE
{'='*80}

TEST SET:
  Baseline: P={baseline_test_summary['Precision']:.4f} R={baseline_test_summary['Recall']:.4f} F1={baseline_test_summary['F1']:.4f}
  ECA:      P={eca_test_summary['Precision']:.4f} R={eca_test_summary['Recall']:.4f} F1={eca_test_summary['F1']:.4f}

VALIDATION SET:
  Baseline: P={baseline_val_summary['Precision']:.4f} R={baseline_val_summary['Recall']:.4f} F1={baseline_val_summary['F1']:.4f}
  ECA:      P={eca_val_summary['Precision']:.4f} R={eca_val_summary['Recall']:.4f} F1={eca_val_summary['F1']:.4f}

RECOMMENDATION: {'ECA' if eca_test_summary['Recall'] > baseline_test_summary['Recall'] else 'Baseline'} for disaster detection (recall priority)

OUTPUTS:
  Excel: {EXCEL_DIR}
  Plots: {PLOT_DIR}
  Reports: {REPORT_DIR}
{'='*80}
"""

with open(f"{REPORT_DIR}/MASTER_SUMMARY.txt", "w") as f:
    f.write(master_report)

print(master_report)

from IPython.display import FileLink
!zip -r /kaggle/working/complete_results.zip {EXCEL_DIR} {PLOT_DIR} {REPORT_DIR}

FileLink("/kaggle/working/complete_results.zip")


FINAL SUMMARY

DISASTER HUMAN DETECTION: ABLATION STUDY COMPLETE

TEST SET:
  Baseline: P=0.8304 R=0.8529 F1=0.8415
  ECA:      P=0.8342 R=0.8458 F1=0.8400

VALIDATION SET:
  Baseline: P=0.8075 R=0.7944 F1=0.8009
  ECA:      P=0.8047 R=0.7967 F1=0.8007

RECOMMENDATION: Baseline for disaster detection (recall priority)

OUTPUTS:
  Excel: /kaggle/working/excel_reports
  Plots: /kaggle/working/plots
  Reports: /kaggle/working/benchmark_reports

  adding: kaggle/working/excel_reports/ (stored 0%)
  adding: kaggle/working/excel_reports/yolo11m_eca_test_detailed.xlsx (deflated 7%)
  adding: kaggle/working/excel_reports/yolo11m_baseline_training.xlsx (deflated 6%)
  adding: kaggle/working/excel_reports/yolo11m_baseline_test_detailed.xlsx (deflated 7%)
  adding: kaggle/working/excel_reports/complete_evaluation_summary.xlsx (deflated 10%)
  adding: kaggle/working/excel_reports/yolo11m_eca_test_calibration.xlsx (deflated 11%)
  adding: kaggle/working/excel_reports/yolo11m_baseline_val_detailed.

/kaggle/working/complete_results.zip